In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import pandas as pd

df = pd.read_csv('..//Data//patch.csv')

train_inputs = df[df['year'] < 2024].iloc[: , :233]
train_target = df[df['year'] < 2024].iloc[: , 233:]
test_inputs = df[df['year'] >= 2024].iloc[: , :233]
test_target = df[df['year'] >= 2024].iloc[: , 233:]

train_country_year = train_inputs.iloc[:,[0,1]]
test_country_year = test_inputs.iloc[:,[0,1]]

train_inputs.drop(['country'], axis=1, inplace=True)
test_inputs.drop(['country'], axis=1, inplace=True)
print(train_inputs)




      year  host  Aquatics  Aquatics_gold  Aquatics_silver  Aquatics_bronze  \
0     1900     0         4              0                0                0   
1     1900     0         4              0                0                0   
2     1900     0         4              0                0                0   
3     1900     0         4              0                0                0   
4     1900     0         4              0                0                0   
...    ...   ...       ...            ...              ...              ...   
6015  2020     0        46              0                0                0   
6016  2020     0        46              0                0                0   
6017  2020     0        46              0                0                0   
6018  2020     0        46              0                0                0   
6019  2020     0        46              0                0                0   

      Aquatics_no  Archery  Archery_gold  Archery_s

In [2]:
train_time_col = train_inputs['year']  # 年份列
train_features = train_inputs.drop(columns=['year'])  # 其他特征
test_time_col = test_inputs['year']
test_features = test_inputs.drop(columns=['year'])


year_min = train_time_col.min()
train_time_indices = train_time_col - year_min
test_time_indices = test_time_col - year_min

embedding_dim = 16
num_years = max(train_time_col.max(), test_time_col.max()) - year_min + 1
time_embedding = nn.Embedding(num_embeddings=num_years, embedding_dim=embedding_dim)

# 嵌入时间特征
embedded_train_time = time_embedding(torch.tensor(train_time_indices.values, dtype=torch.long))
embedded_test_time = time_embedding(torch.tensor(test_time_indices.values, dtype=torch.long))

train_x = torch.tensor(train_features.values, dtype=torch.float32)
test_x = torch.tensor(test_features.values, dtype=torch.float32)
train_x = torch.cat([train_x, embedded_train_time], dim=1)
test_x = torch.cat([test_x, embedded_test_time], dim=1)



seq_length = 1


train_x = train_x.view(-1, seq_length, train_x.shape[1])
test_x = test_x.view(-1, seq_length, test_x.shape[1])
# 调整 train_y 的形状



# 转换目标为张量
train_y = torch.tensor(train_target.values, dtype=torch.float32)
test_y = torch.tensor(test_target.values, dtype=torch.float32)

In [3]:
class StackedLSTMWithAttention(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size, dropout_prob):
        super(StackedLSTMWithAttention, self).__init__()
        
        # 堆叠多层 LSTM
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout_prob)
        
        # Attention 层
        self.attention = nn.Linear(hidden_size, 1)
        
        # 全连接层
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        # 初始化隐藏状态和记忆单元
        h0 = torch.zeros(self.lstm.num_layers, x.size(0), self.lstm.hidden_size)
        c0 = torch.zeros(self.lstm.num_layers, x.size(0), self.lstm.hidden_size)
        
        # LSTM 输出
        lstm_out, _ = self.lstm(x)  # lstm_out: [batch_size, seq_length, hidden_size]
        
        # Attention 机制
        attn_weights = torch.softmax(self.attention(lstm_out), dim=1)  # [batch_size, seq_length, 1]
        context = torch.sum(attn_weights * lstm_out, dim=1)  # 加权求和, [batch_size, hidden_size]
        
        # 全连接层输出
        out = self.fc(context)  # [batch_size, output_size]
        return out

In [4]:
input_size = train_x.size(-1) # 特征数
hidden_size = 64  # 隐藏层单元数（可调整）
num_layers = 2  # 堆叠 LSTM 层数
output_size = 3  # 输出目标数（目标的列数）
dropout_prob = 0.3  # Dropout 概率


model = StackedLSTMWithAttention(input_size, hidden_size, num_layers, output_size, dropout_prob)



In [5]:
criterion = nn.MSELoss()  # 仍然使用均方误差，因为这是一个回归问题
optimizer = optim.Adam(model.parameters(), lr=0.01)

num_epochs = 5000

for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(train_x)  # 前向传播
    

    
    loss = criterion(outputs, train_y)  # 计算损失
    loss.backward(retain_graph=True)  # 反向传播
    optimizer.step()  # 更新参数
    
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.4f}')
        
        



Epoch [10/5000], Loss: 11.4818
Epoch [20/5000], Loss: 8.9240
Epoch [30/5000], Loss: 7.4261
Epoch [40/5000], Loss: 6.3764
Epoch [50/5000], Loss: 5.5657
Epoch [60/5000], Loss: 5.0646
Epoch [70/5000], Loss: 4.7436
Epoch [80/5000], Loss: 4.5479
Epoch [90/5000], Loss: 4.4043
Epoch [100/5000], Loss: 4.1619
Epoch [110/5000], Loss: 4.0527
Epoch [120/5000], Loss: 3.9879
Epoch [130/5000], Loss: 3.6738
Epoch [140/5000], Loss: 3.8024
Epoch [150/5000], Loss: 3.6819
Epoch [160/5000], Loss: 3.3699
Epoch [170/5000], Loss: 3.5085
Epoch [180/5000], Loss: 3.5091
Epoch [190/5000], Loss: 3.3567
Epoch [200/5000], Loss: 3.1830
Epoch [210/5000], Loss: 3.2592
Epoch [220/5000], Loss: 3.1391
Epoch [230/5000], Loss: 3.1409
Epoch [240/5000], Loss: 3.1850
Epoch [250/5000], Loss: 3.1273
Epoch [260/5000], Loss: 2.9770
Epoch [270/5000], Loss: 3.2623
Epoch [280/5000], Loss: 2.9539
Epoch [290/5000], Loss: 2.9593
Epoch [300/5000], Loss: 2.9083
Epoch [310/5000], Loss: 3.0599
Epoch [320/5000], Loss: 2.9886
Epoch [330/5000]

KeyboardInterrupt: 

In [ ]:
model.eval()
with torch.no_grad():
    predicted = model(test_x).detach().numpy()  # 预测值
    rounded_outputs = torch.round(outputs)
    rounded_outputs = torch.clamp(rounded_outputs, min=0)  # Clamp all values to be >= 0
    


predicted_df = pd.DataFrame(predicted)
test_country_year.reset_index(drop = True,inplace=True)
predicted_df.reset_index(drop = True,inplace=True)

output = pd.concat([test_country_year, predicted_df],axis=1)
print(output)

output.to_csv('..//output//output.csv', index=False)

               country  year         0         1         2
0          Afghanistan  2024  0.046631  0.140964  0.234294
1              Albania  2024  0.027085  0.079508  0.128525
2              Algeria  2024  0.252694  0.212679  0.442460
3              Andorra  2024  0.050595  0.144354  0.218078
4               Angola  2024  0.421328  0.340997  0.467134
..                 ...   ...       ...       ...       ...
425         Yugoslavia  2028  0.032592  0.068224  0.088433
426  Yugoslaviaoslavia  2028  0.032592  0.068224  0.088433
427             Zambia  2028  0.035016  0.058857  0.072410
428           Zimbabwe  2028  0.032657  0.068231  0.088420
429        the Solomon  2028  0.032573  0.068217  0.088414

[430 rows x 5 columns]


In [9]:
import shap
import torch
import numpy as np


# 选择一些测试数据
# 注意：SHAP中的输入数据需要转换成合适的格式
background_data = train_x[:100]  # 使用前100个样本作为背景数据（用于估算Shapley值）
test_data = test_x[:10]  # 选择10个样本来进行测试

# 创建SHAP解释器
explainer = shap.DeepExplainer(model, background_data)

# 计算Shapley值
shap_values = explainer.shap_values(test_data)

# 查看Shapley值的形状
print("Shapley values shape:", np.array(shap_values).shape)


e:\Users\lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
e:\Users\lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\shap\explainers\_deep\deep_pytorch.py:243: UserWarning: unrecognized nn.Module: LSTM
  warnings.warn(f'unrecognized nn.Module: {module_type}')


AssertionError: The SHAP explanations do not sum up to the model's output! This is either because of a rounding error or because an operator in your computation graph was not fully supported. If the sum difference of %f is significant compared to the scale of your model outputs, please post as a github issue, with a reproducible example so we can debug it. Used framework: pytorch - Max. diff: 1.92786630205228 - Tolerance: 0.01